## Proof of Concept for flood maps

This notebook is an attempt at translating the CoCliCo User Story into code. 

The data used is Coastal Flood Hazard Projections and can be found here: p:\11207608-coclico\FULLTRACK_DATA\WP4\

In [1]:
# Import modules

import warnings
import traceback
# import holoviews as hv
import cartopy.crs as ccrs
import cartopy.feature as cf
import matplotlib.pyplot as plt
import matplotlib.ticker as tck
import numpy as np
import shapely
import pandas as pd
import pystac_client
import xarray as xr
import rioxarray as rio
import pathlib
from pathlib import Path
import pystac
import pystac_client
import geopandas as gpd
from shapely.geometry import shape
from pystac.extensions.projection import ProjectionExtension
from tqdm import tqdm
import rasterio

#import colormaps as cmaps
import matplotlib.colors as mcolors

from copy import deepcopy
from typing import List, Dict
from rioxarray.merge import merge_arrays

import damagescanner.download as download
from damagescanner.core import DamageScanner
from damagescanner.osm import read_osm_data
from damagescanner.config import DICT_CIS_VULNERABILITY_FLOOD

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning) # exactextract gives a warning that is invalid

In [3]:
# List all files in a directory
folder_path = Path(r'X:\eks510\MIRACA_results')
# Just filenames
filenames = [f.name for f in folder_path.iterdir() if f.is_file()]
print(filenames)

['GRC_healthcare_coastal_2100_SSP585_risk.parquet', 'HRV_education_coastal_2050_SSP585_risk.parquet', 'IRL_education_windstorm_risk.parquet', 'HRV_rail_coastal_2050_SSP585_risk.parquet', 'EST_rail_coastal_2050_SSP585_risk.parquet', 'MLT_rail_coastal_2100_SSP245_risk.parquet', 'PRT_roads_coastal_2100_SSP245_risk.parquet', 'ITA_power_river_risk.parquet', 'POL_healthcare_windstorm_risk.parquet', 'EST_education_river_risk.parquet', 'HRV_healthcare_windstorm_risk.parquet', 'DEU_rail_coastal_risk.parquet', 'EST_rail_coastal_2100_SSP245_risk.parquet', 'HRV_rail_coastal_2100_SSP245_risk.parquet', 'HRV_education_coastal_2100_SSP245_risk.parquet', 'GRC_healthcare_coastal_2050_SSP245_risk.parquet', 'LTU_roads_coastal_risk.parquet', 'GRC_rail_coastal_risk.parquet', 'PRT_roads_coastal_2050_SSP585_risk.parquet', 'MLT_rail_coastal_2050_SSP585_risk.parquet', 'BGR_education_river_risk.parquet', 'PRT_education_coastal_risk.parquet', 'LUX_healthcare_river_risk.parquet', 'POL_education_coastal_2100_SSP585

## Open STAC data

1st connect to the catalog, 
2nd retrieve the collection of interest, 
3rd retrieve one item to see it's contents

In [2]:
# Setup the URL to STAC catalog in Google Cloud
catalog = pystac_client.Client.open(
    "https://storage.googleapis.com/coclico-data-public/coclico/coclico-stac-23jan25/catalog.json"
)

# Retrieve collection from catalog, in this case Coastal Flood Hazard Projections (cfhp)
collection = catalog.get_child(id = 'cfhp_all')

C:\Users\eks510\.conda\envs\pygis\Lib\site-packages\pystac_client\client.py:191: NoConformsTo: Server does not advertise any conformance classes.
  warnings.warn(NoConformsTo())


In [3]:
collection

<CollectionClient id=cfhp_all>

In [ ]:
# Retrieve a single item for testing
flood_item = collection.get_item(r'UNDEFENDED_MAPS\RP\1000.tif')
flood_item = collection.get_item(r'HIGH_DEFENDED_MAPS\SLR\SSP585\2100.tif')

# Show test item
flood_item

C:\Users\eks510\.conda\envs\pygis\Lib\site-packages\pystac_client\collection_client.py:201: FallbackToPystac: Falling back to pystac. This might be slow.
  root._warn_about_fallback("FEATURES", "ITEM_SEARCH")
C:\Users\eks510\.conda\envs\pygis\Lib\site-packages\pystac_client\collection_client.py:153: FallbackToPystac: Falling back to pystac. This might be slow.
  root._warn_about_fallback("ITEM_SEARCH")


In [ ]:
flood_item

In [5]:
country_full_name = 'Portugal'
country_iso3 = 'PRT'

In [6]:
building_data_path = all_building_paths[-1]

NameError: name 'all_building_paths' is not defined

In [ ]:
def building_exposure(building_data_path):

    # load building data
    buildings = gpd.read_parquet(building_data_path)
    
    # if flood_col in buildings.columns:
    #     return buildings
        
    # set crs
    #buildings = buildings.to_crs(3035)
    
    #create centroids
    buildings['centroid'] = buildings.geometry.centroid
    
    #build tree 
    building_tree = shapely.STRtree(buildings.geometry)

    for flood_item in list(collection.get_items()):
                
        flood_name = '_'.join(flood_item.id.split('\\')).split('.')[0]
           
        # run through the flood maps and overlay the  centroids
        collect_overlays = {}
        for iter_,flood_map in tqdm(enumerate(flood_item.assets),
                                    total=len(flood_item.assets),
                                   desc=str(building_data_path).split('\\')[-1][:3]+' '+flood_name):
            if iter_ == 0:
                continue
        
            # get the flood chunk
            flood_chunk = flood_item.assets.get(flood_map)
        
            # Access the Projection extension on the asset
            projection = ProjectionExtension.ext(flood_chunk)
        
            # create a bbox and check if the buildings are within that bbox. 
            #If no overlap, we continue to the next hazard map and dont even load it
            [chunk_bbox] = projection.geometry['coordinates']
            chunk_geom = shapely.Polygon(chunk_bbox)
            
            if len(building_tree.query(chunk_geom)) == 0:
                continue
        
            # open hazard data
            hazard = rasterio.open(flood_chunk.href)
        
            # perform overlay
            values = np.array(
                            [
                                value[0]
                                for value in rasterio.sample.sample_gen(
                                    hazard,
                                    [
                                        (point.x, point.y)
                                        for point in buildings.centroid
                                    ],
                                )
                            ]
                        )   
            collect_overlays[iter_] = values
        
        # find the maximum value across the different flood chunks that overlay with the building data (should
        # just return 1 value per building, as probably the other hazard maps provide NaN values
        flood_overlay = pd.DataFrame(pd.DataFrame.from_dict(collect_overlays))
        flood_overlay = flood_overlay.replace(-9999,np.nan).max(axis=1)
        try:
            buildings [flood_name] = flood_overlay.values
        except:
            print(str(building_data_path).split('\\')[-1][:3]+' '+flood_name+' failed!')

    return buildings
    
    # # merge and save: NOTE: this is not ideal, but helps for now to get this running
    #buildings.to_parquet(building_data_path)
    
    # return buildings


In [ ]:
%%time
for building_data_path in all_building_paths:

    if 'SWE' not in str(building_data_path):
        continue
    
    buildings = building_exposure(building_data_path,flood_item,flood_col='flood_depth_1_1000')

    buildings.drop(['geometry'],axis=1).rename(columns={"centroid": "geometry"}).to_parquet(output_path / str(building_data_path).split('\\')[-1])
    
    try:
        buildings.drop(['centroid','flood_depth_1_1000'],axis=1).to_parquet(building_data_path,compression=None)
    except:
        continue


In [4]:
%%time

# load data
path_to_population = Path("C:\\Data\\Population\\SSP1_2010_EU_UK.tif")
pop_data = xr.open_dataset(path_to_population)

# convert to dataframe
df_pop = pop_data.to_dataframe()
df_pop = df_pop[df_pop.band_data.notna()].reset_index()

#create geometry values and drop lat lon columns
df_pop['geometry'] = gpd.points_from_xy(df_pop.x, df_pop.y, crs="EPSG:4326")
df_pop = df_pop.drop(['x','y','band','spatial_ref'],axis=1)

# get the resolution
resolution = abs(pop_data.rio.resolution()[0]) 

# and turn them into squares again:
df_pop.geometry= shapely.buffer(df_pop.geometry,
                                          distance=resolution/2,cap_style='square').values

CPU times: total: 1min 8s
Wall time: 1min 11s


In [ ]:
%%time
for flood_item in list(collection.get_items()):
                
    flood_name = '_'.join(flood_item.id.split('\\')).split('.')[0]

    file_path = output_heatmap / f"{flood_name}_buildings_flooded.tif"
    
    if file_path.exists():
        continue  # Skip to the next iteration if file exists

    
    collect_countries = {}
    for building_data_path in tqdm(all_exposure_paths,total=len(all_exposure_paths),desc=flood_name):
    
        iso3= str(building_data_path).split('\\')[-1][:3]
        
        # load building data
        buildings = gpd.read_parquet(building_data_path)
        
        # set crs
        buildings = buildings.to_crs(4326)
    
        # build tree totals
        building_tree = shapely.STRtree(buildings.geometry)
        overlay_buildings = building_tree.query(df_pop.geometry,predicate='intersects')
        df_total = pd.DataFrame(overlay_buildings).T
        df_total.columns = ['pop_cells','buildings']
        df_total_buildings = df_total.groupby('pop_cells').count()
    
        # build tree flooded
        try:
            flooded_buildings = buildings[buildings[flood_name].notna()].reset_index(drop=True)
            flooded_building_tree = shapely.STRtree(flooded_buildings.geometry)
            overlay_flooded_buildings = flooded_building_tree.query(df_pop.geometry,predicate='intersects')
            df_flooded_total = pd.DataFrame(overlay_flooded_buildings).T
            df_flooded_total.columns = ['pop_cells','flooded_buildings']
            df_flooded_buildings = df_flooded_total.groupby('pop_cells').count()
        
            cell_buildings = df_total_buildings.merge(df_flooded_buildings,left_index=True,right_index=True,how='outer').fillna(0)
            cell_buildings = gpd.GeoDataFrame(df_pop.merge(cell_buildings,left_index=True,right_index=True))
            collect_countries[iso3] = cell_buildings
        except Exception as e: 
            print(e)
            continue

    
    exposure_cells = pd.concat(collect_countries).reset_index().set_index('level_1')
    exposure_cells['geometry'] = exposure_cells.centroid
    exposure_cells['perc_flooded'] = (exposure_cells.flooded_buildings/exposure_cells.buildings)*100
    
    # Open example raster
    raster = rasterio.open(path_to_population)
    
    # create tuples of geometry, value pairs, where value is the attribute value you want to burn
    geom_value = ((geom,value) for geom, value in zip(exposure_cells.geometry, exposure_cells['flooded_buildings']))
    
    # Rasterize vector using the shape and transform of the raster
    rasterized = rasterio.features.rasterize(geom_value,
                                    out_shape = raster.shape,
                                    transform = raster.transform,
                                    all_touched = True,
                                    fill = -9999,   # background value
                                    merge_alg = rasterio.enums.MergeAlg.replace,
                                    dtype = rasterio.float32)
    
    with rasterio.open(
            output_heatmap / f"{flood_name}_buildings_flooded.tif", "w",
            driver = "COG", compress="DEFLATE",
            crs = raster.crs,
            transform = raster.transform,
            dtype = rasterio.float32,
            count = 1,
            width = raster.width,
            height = raster.height) as dst:
        dst.write(rasterized, indexes = 1)

LOW_DEFENDED_MAPS_1000_SSP126_2100:  40%|██████████████████▍                           | 10/25 [01:55<02:58, 11.88s/it]